In [1]:
import pandas as pd
import numpy as np
import os

In [5]:
RAW_PATH = "../data/raw/"
CLEANED_PATH = "../data/processed/"
os.makedirs(CLEANED_PATH, exist_ok=True)

**Load Raw Datasets**

In [6]:
customer_data = pd.read_csv(
    RAW_PATH + "olist_customers_dataset.csv"
)

order_data = pd.read_csv(
    RAW_PATH + "olist_orders_dataset.csv"
)

order_items_data = pd.read_csv(
    RAW_PATH + "olist_order_items_dataset.csv"
)

payment_data = pd.read_csv(
    RAW_PATH + "olist_order_payments_dataset.csv"
)

review_data = pd.read_csv(
    RAW_PATH + "olist_order_reviews_dataset.csv"
)

product_data = pd.read_csv(
    RAW_PATH + "olist_products_dataset.csv"
)

seller_data = pd.read_csv(
    RAW_PATH + "olist_sellers_dataset.csv"
)

category_translation = pd.read_csv(
    RAW_PATH + "product_category_name_translation.csv"
)

**Validate Dataset**

In [15]:
def data_quality_check(df, name):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nData types:")
    print(df.dtypes)
    print("Descriptive statistics:")
    print(df.describe(include='all'))

print(data_quality_check(customer_data, "Customers"))
print(data_quality_check(order_data, "Orders"))
print(data_quality_check(order_items_data, "Order Items"))
print(data_quality_check(payment_data, "Payments"))
print(data_quality_check(review_data, "Reviews"))
print(data_quality_check(product_data, "Products"))
print(data_quality_check(seller_data, "Sellers"))
print(data_quality_check(category_translation, "Category Translation"))

--- Customers ---
Shape: (99441, 5)
Duplicate rows: 0

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
Descriptive statistics:
                             customer_id                customer_unique_id  \
count                              99441                             99441   
unique                             99441                             96096   
top     06b8999e2fba1a1fbc88172c00ba8bc7  8d50f5eadf50201ccdcedfb9e2ac8455   
freq                                   1                                17   
mean                                 NaN                               NaN   
std                                  NaN                               NaN   

**Clean Customer**

In [16]:
customer_data["customer_id"] = (
    customer_data["customer_id"].astype("string")
)

customer_data["customer_unique_id"] = (
    customer_data["customer_unique_id"].astype("string")
)

In [17]:
customer_data["customer_city"] = (
    customer_data["customer_city"]
    .str.strip()
    .str.lower()
)

customer_data["customer_state"] = (
    customer_data["customer_state"]
    .str.strip()
    .str.upper()
)

In [19]:
print(customer_data.isnull().sum())
print(customer_data.duplicated().sum())

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
0


**Clean Orders**

In [21]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

order_data[order_date_columns] = (
    order_data[order_date_columns]
    .apply(pd.to_datetime, format="%d-%m-%Y %H:%M")
)

In [22]:
order_data["order_purchase_year"] = (
    order_data["order_purchase_timestamp"].dt.year
)

order_data["order_purchase_month"] = (
    order_data["order_purchase_timestamp"].dt.month
)

order_data["order_purchase_month_name"] = (
    order_data["order_purchase_timestamp"].dt.month_name()
)

In [23]:
order_data["order_delivery_time"] = (
    order_data["order_delivered_customer_date"]
    - order_data["order_purchase_timestamp"]
).dt.days

In [24]:
order_data["delivery_delay_days"] = (
    order_data["order_delivered_customer_date"]
    - order_data["order_estimated_delivery_date"]
).dt.days

In [25]:
order_data["delivery_status"] = np.select(
    [
        order_data["delivery_delay_days"] > 0,
        order_data["delivery_delay_days"] == 0,
        order_data["delivery_delay_days"] < 0
    ],
    [
        "Late",
        "On Time",
        "Early"
    ],
    default="Unknown"
)

In [26]:
order_data["order_id"].duplicated().sum()

np.int64(0)

**Clean Order-items**

In [27]:
order_items_data.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [34]:
order_items_data["price"] = pd.to_numeric(
    order_items_data["price"],
    errors="coerce"
)

order_items_data["freight_value"] = pd.to_numeric(
    order_items_data["freight_value"],
    errors="coerce"
)

In [35]:
order_items_data["shipping_limit_date"] = pd.to_datetime(
    order_items_data["shipping_limit_date"]
)

In [36]:
(order_items_data["price"] < 0).sum()
(order_items_data["freight_value"] < 0).sum()

np.int64(0)

**Clean Payment**

In [37]:
payment_data["payment_sequential"] = pd.to_numeric(
    payment_data["payment_sequential"],
    errors="coerce"
)

payment_data["payment_installments"] = pd.to_numeric(
    payment_data["payment_installments"],
    errors="coerce"
)

payment_data["payment_value"] = pd.to_numeric(
    payment_data["payment_value"],
    errors="coerce"
)

In [38]:
payment_data["payment_type"] = (
    payment_data["payment_type"]
    .str.strip()
    .str.lower()
)

In [40]:
payment_data.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

np.int64(0)

**Clean Reviews**

In [41]:
review_data["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [42]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

review_data[review_date_columns] = (
    review_data[review_date_columns]
    .apply(pd.to_datetime)
)

In [43]:
review_data["review_type"] = np.where(
    review_data["review_score"] <= 3,
    "Negative",
    "Positive"
)

In [44]:
review_data["review_id"].duplicated().sum()

np.int64(814)

**Clean Products**

In [45]:
product_data["product_category_name"] = (
    product_data["product_category_name"]
    .str.strip()
    .str.lower()
)

In [46]:
product_data["product_category_name"].isnull().sum()

np.int64(610)

In [47]:
product_data["product_category_name"] = (
    product_data["product_category_name"]
    .fillna("unknown")
)

In [48]:
category_translation["product_category_name"] = (
    category_translation["product_category_name"]
    .str.strip()
    .str.lower()
)

product_data = product_data.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

**Clean Sellers**

In [50]:
seller_data["seller_city"] = (
    seller_data["seller_city"]
    .str.strip()
    .str.lower()
)

seller_data["seller_state"] = (
    seller_data["seller_state"]
    .str.strip()
    .str.upper()
)

In [52]:
print(seller_data["seller_id"].duplicated().sum())
print(seller_data.isnull().sum())

0
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


## Final Validation

In [54]:
print(data_quality_check(customer_data, "Customers"))
print(data_quality_check(order_data, "Orders"))
print(data_quality_check(order_items_data, "Order Items"))
print(data_quality_check(payment_data, "Payments"))
print(data_quality_check(review_data, "Reviews"))
print(data_quality_check(product_data, "Products"))
print(data_quality_check(seller_data, "Sellers"))

--- Customers ---
Shape: (99441, 5)
Duplicate rows: 0

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Data types:
customer_id                 string
customer_unique_id          string
customer_zip_code_prefix     int64
customer_city                  str
customer_state                 str
dtype: object
Descriptive statistics:
                             customer_id                customer_unique_id  \
count                              99441                             99441   
unique                             99441                             96096   
top     06b8999e2fba1a1fbc88172c00ba8bc7  8d50f5eadf50201ccdcedfb9e2ac8455   
freq                                   1                                17   
mean                                 NaN                               NaN   
std                                  NaN                               N

In [56]:
print("Customers:", customer_data["customer_id"].nunique())
print("Orders:", order_data["order_id"].nunique())
print("Products:", product_data["product_id"].nunique())
print("Sellers:", seller_data["seller_id"].nunique())
print(
    order_data["customer_id"]
    .isin(customer_data["customer_id"])
    .mean()
)

Customers: 99441
Orders: 99441
Products: 32951
Sellers: 3095
1.0


## Export Cleaned Tables

In [57]:
customer_data.to_csv(
    CLEANED_PATH + "customers_cleaned.csv",
    index=False
)

order_data.to_csv(
    CLEANED_PATH + "orders_cleaned.csv",
    index=False
)

order_items_data.to_csv(
    CLEANED_PATH + "order_items_cleaned.csv",
    index=False
)

payment_data.to_csv(
    CLEANED_PATH + "payments_cleaned.csv",
    index=False
)

review_data.to_csv(
    CLEANED_PATH + "reviews_cleaned.csv",
    index=False
)

product_data.to_csv(
    CLEANED_PATH + "products_cleaned.csv",
    index=False
)

seller_data.to_csv(
    CLEANED_PATH + "sellers_cleaned.csv",
    index=False
)